In [0]:
import os
import sys
from pyspark.sql.functions import current_timestamp, input_file_name, col, lit

parent_path = '/Workspace/Users/esymphony.life@gmail.com/MSF_test'
if parent_path not in sys.path:
    sys.path.insert(0, parent_path)

from customer.common.functions import get_read_options

In [0]:

def get_uningested_files(volume_path, schema="bronze"):
    """
    Identifies all files in a Volume that have not yet been ingested into 
    the corresponding table (derived from the volume path).
    """
    # Parse volume_path to derive the catalog.schema.table
    _, _, catalog, domain, table_name = volume_path.split("/")
    target_table = f"{catalog}.{domain}.{schema}_{table_name}"   

    # Get current files in the volume
    unprocessed_df = (spark.read.format("binaryFile")
                        .option("recursiveFileLookup", "true")
                        .load(volume_path)
                        .select(col("path"), col("modificationTime")))

    # Check against the derived target table
    if spark.catalog.tableExists(target_table):
        # Select distinct source files already processed
        ingested_df = (spark.read.table(target_table)
                       .select(col("_source_file").alias("path"))
                       .distinct())
        
        # Filter out already processed files
        unprocessed_df = unprocessed_df.join(ingested_df, on="path", how="left_anti")

    unprocessed_rows = (unprocessed_df
                        .orderBy(col("modificationTime").asc())
                        .select("path")
                        .collect())

    # Return a list of strings
    return [row.path[5:] for row in unprocessed_rows]


In [0]:
def ingest_to_delta(spark, filetype, file_path, table_path, read_options):
    """
    Ingest any file type to Delta table using Auto Loader

    Args:
        filetype: File format (json, csv, parquet, avro, etc.)
        file_path: Source file path or pattern (can be volume path with wildcards)
        table_path: Target Delta table name (catalog.schema.table)
        read_options: Optional dict of read options (e.g., {"multiLine": True, "header": True})
    """
    # Extract base volume path from file_path (remove wildcards and filename)
    # Example: /Volumes/catalog/schema/volume/*.csv -> /Volumes/catalog/schema/volume
    base_volume = file_path.split('*')[0].rstrip('/')
    
    # Parse table name for checkpoint subdirectory
    table_name = table_path.split('.')[-1]
    
    # Use a _checkpoints subdirectory within the source volume
    checkpoint_base = f"{base_volume}/_checkpoints/{table_name}"
    schema_location = f"{checkpoint_base}/schema"
    checkpoint_location = f"{checkpoint_base}/ckpt"
    
    # Create checkpoint directories if they don't exist
    dbutils.fs.mkdirs(schema_location)
    dbutils.fs.mkdirs(checkpoint_location)

    if read_options:
        df = spark.readStream\
            .format("cloudFiles")\
            .option("cloudFiles.format", filetype)\
            .option("cloudFiles.schemaLocation", schema_location)\
            .options(**read_options)\
            .load(file_path)
    else:
        df = spark.readStream\
            .format("cloudFiles")\
            .option("cloudFiles.format", filetype)\
            .option("cloudFiles.schemaLocation", schema_location)\
            .load(file_path)

    # Use _metadata.file_path for Unity Catalog compatibility (instead of input_file_name())
    df = df.withColumn("_ingest_timestamp", current_timestamp())\
        .withColumn("_source_file", col("_metadata.file_path"))

    df.writeStream\
        .format("delta")\
        .option("mergeSchema", "true")\
        .option("checkpointLocation", checkpoint_location)\
        .trigger(availableNow=True)\
        .toTable(table_path)

    return df

In [0]:
def ingest_volume(vol_path):
    _, _, catalog, domain, volume = vol_path.split("/")
    schema = "bronze"
    write_path = f"{catalog}.{domain}.{schema}_{volume}"

    # Discover all file types in the volume
    all_files = (spark.read.format("binaryFile")
                 .option("recursiveFileLookup", "true")
                 .load(vol_path)
                 .select(col("path"))
                 .collect())

    # Find unique file extensions
    file_types = set()
    for row in all_files:
        filetype = row.path.split(".")[-1]
        file_types.add(filetype)

    # Ingest
    for filetype in file_types:
        read_options = get_read_options(filetype)
        file_pattern = f"{vol_path}/*.{filetype}"
        ingest_to_delta(spark, filetype, file_pattern, write_path, read_options)

    print(f"Successfully ingested all files from {volume}")

In [0]:
ingest_volume(dbutils.widgets.get("VOLUME_PATH"))